# Обучение Item2Vec

Здесь обучаются три Word2Vec-модели по последовательностям товаров: отдельно для `clicks`, `carts` и `orders`. Модели сохраняются в `/kaggle/working/item2vec_{clicks,carts,orders}.model`.

Для обучения используются события до начала последней недели данных. Последняя неделя остаётся для validation.


In [1]:
!pip install faiss-cpu --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 66.4 MB/s eta 0:00:00


In [2]:
import csv
import gc
import json
import math
from collections import defaultdict
from datetime import datetime, timedelta, timezone
from pathlib import Path

import faiss
import numpy as np
from gensim.models import Word2Vec

TRAIN_PATH = '/kaggle/input/competitions/otto-recommender-system/train.jsonl'
OUT_DIR = Path('/kaggle/working')
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_END = datetime(2022, 8, 28, 21, 59, 59, 984000, tzinfo=timezone.utc)
VALIDATION_DAYS = 7
VALIDATION_START = DATA_END - timedelta(days=VALIDATION_DAYS)
VALIDATION_START_TS = int(VALIDATION_START.timestamp() * 1000)
EVENT_TYPES = ('clicks', 'carts', 'orders')


## Подготовка последовательностей

Из каждой сессии в отдельный поток записываются последовательности `aid` по типам событий. В файл обучения попадают только события раньше границы validation; сами последовательности читаются из файла по мере обучения.


In [3]:
session_files = {event_type: OUT_DIR / f'sessions_{event_type}.txt' for event_type in EVENT_TYPES}

train_event_count = 0
validation_event_count = 0
train_sessions_count = 0
validation_sessions_count = 0
sessions_crossing_boundary = 0
total_sessions = 0

with open(TRAIN_PATH, 'r') as train_file, open(session_files['clicks'], 'w') as clicks_file, \
     open(session_files['carts'], 'w') as carts_file, open(session_files['orders'], 'w') as orders_file:
    output_files = {
        'clicks': clicks_file,
        'carts': carts_file,
        'orders': orders_file,
    }

    for session_index, line in enumerate(train_file):
        total_sessions += 1
        if session_index % 1_000_000 == 0:
            print(f'Обработано записей-сессий: {session_index:,}')

        events = json.loads(line).get('events', [])
        by_type = defaultdict(list)
        session_has_train_events = False
        session_has_validation_events = False

        for event in events:
            if int(event['ts']) >= VALIDATION_START_TS:
                validation_event_count += 1
                session_has_validation_events = True
                continue

            train_event_count += 1
            session_has_train_events = True
            event_type = event['type']
            if event_type in output_files:
                by_type[event_type].append(str(event['aid']))

        if session_has_train_events:
            train_sessions_count += 1
        if session_has_validation_events:
            validation_sessions_count += 1
        if session_has_train_events and session_has_validation_events:
            sessions_crossing_boundary += 1

        for event_type, output_file in output_files.items():
            sequence = by_type[event_type]
            if sequence:
                output_file.write(' '.join(sequence) + '\n')

print(f'Всего записей-сессий: {total_sessions:,}')
print(f'Train: {train_sessions_count:,} сессий, {train_event_count:,} событий')
print(f'Validation: {validation_sessions_count:,} сессий, {validation_event_count:,} событий')
print(f'Сессий с событиями по обе стороны границы: {sessions_crossing_boundary:,}')
print(f'Validation начинается: {VALIDATION_START.isoformat()}')
print('Последовательности подготовлены:', {k: str(v) for k, v in session_files.items()})


Обработано записей-сессий: 0
Обработано записей-сессий: 1,000,000
Обработано записей-сессий: 2,000,000
Обработано записей-сессий: 3,000,000
Обработано записей-сессий: 4,000,000
Обработано записей-сессий: 5,000,000
Обработано записей-сессий: 6,000,000
Обработано записей-сессий: 7,000,000
Обработано записей-сессий: 8,000,000
Обработано записей-сессий: 9,000,000
Обработано записей-сессий: 10,000,000
Обработано записей-сессий: 11,000,000
Обработано записей-сессий: 12,000,000
Всего записей-сессий: 12,899,779
Train: 11,098,528 сессий, 163,955,180 событий
Validation: 5,323,084 сессий, 52,760,916 событий
Сессий с событиями по обе стороны границы: 3,521,833
Validation начинается: 2022-08-21T21:59:59.984000+00:00
Последовательности подготовлены: {'clicks': '/kaggle/working/sessions_clicks.txt', 'carts': '/kaggle/working/sessions_carts.txt', 'orders': '/kaggle/working/sessions_orders.txt'}


## Обучение и сохранение моделей

Обучение идёт последовательно: в памяти находится текущая модель, а после сохранения объект освобождается перед переходом к следующему типу события. Полные модели сохраняются через `Word2Vec.save()` в `/kaggle/working`; созданные Gensim дополнительные файлы остаются рядом с `.model`. 

In [4]:
class SessionReader:
    def __init__(self, filepath):
        self.filepath = filepath

    def __iter__(self):
        with open(self.filepath, 'r') as session_file:
            for line in session_file:
                yield line.split()

configs = {
    'clicks': {'vector_size': 64, 'window': 10, 'min_count': 5, 'epochs': 2, 'workers': 1},
    'carts':  {'vector_size': 64, 'window': 8,  'min_count': 3, 'epochs': 4, 'workers': 1},
    'orders': {'vector_size': 64, 'window': 6,  'min_count': 2, 'epochs': 4, 'workers': 1},
}

for event_type, config in configs.items():
    print(f'Обучение Item2Vec: {event_type}')
    model = Word2Vec(
        sentences=SessionReader(session_files[event_type]),
        sg=1,
        negative=5,
        sample=1e-5,
        seed=11,
        **config,
    )
    model_path = OUT_DIR / f'item2vec_{event_type}.model'
    model.save(str(model_path))
    if not model_path.is_file():
        raise FileNotFoundError(f'Файл модели не создан: {model_path}')

    saved_files = sorted(OUT_DIR.glob(f'{model_path.name}*'))
    print(f'Сохранена модель {event_type}:')
    for saved_file in saved_files:
        print(f'  {saved_file}')

    del model
    gc.collect()
    session_files[event_type].unlink(missing_ok=True)

print('Все три модели обучены и сохранены в /kaggle/working.')

Обучение Item2Vec: clicks
Сохранена модель clicks:
  /kaggle/working/item2vec_clicks.model
  /kaggle/working/item2vec_clicks.model.syn1neg.npy
  /kaggle/working/item2vec_clicks.model.wv.vectors.npy
Обучение Item2Vec: carts
Сохранена модель carts:
  /kaggle/working/item2vec_carts.model
  /kaggle/working/item2vec_carts.model.syn1neg.npy
  /kaggle/working/item2vec_carts.model.wv.vectors.npy
Обучение Item2Vec: orders
Сохранена модель orders:
  /kaggle/working/item2vec_orders.model
  /kaggle/working/item2vec_orders.model.syn1neg.npy
  /kaggle/working/item2vec_orders.model.wv.vectors.npy
Все три модели обучены и сохранены в /kaggle/working.


## Оценка Item2Vec

Для train внутри каждой train-части сессии первые 80% событий используются как история, оставшиеся 20% — как целевые события. Этот score является диагностикой на данных обучения: модели обучены на всех train-событиях. Для train оценивается не более 100 000 сессий.

Для validation история содержит события с timestamp раньше границы, целевые события — события последних семи суток. Для `clicks` целевым считается только следующий click после границы: в разметке OTTO для этого типа действия у сессии один целевой товар. Для `carts` и `orders` — все уникальные товары после границы. В расчёт входят все сессии с событиями в validation.

Считается Recall@20 отдельно для `clicks`, `carts` и `orders`, затем взвешенный итог: `0.10 × clicks + 0.30 × carts + 0.60 × orders`.

In [5]:
import csv
import math
import faiss
import numpy as np
from collections import defaultdict

TOP_K = 20
LAST_N = 10
MAX_TRAIN_EVAL_SESSIONS = 100_000
EVAL_PROGRESS_EVERY = 25_000
EVENT_WEIGHTS = {'clicks': 0.10, 'carts': 0.30, 'orders': 0.60}
ACTION_WEIGHTS = {'clicks': 1.0, 'carts': 3.0, 'orders': 6.0}


def build_item2vec_index(model, M=16, ef_construction=200, ef_search=128):
    vectors = model.wv.vectors
    if vectors.dtype != np.float32:
        vectors = vectors.astype(np.float32)
    faiss.normalize_L2(vectors)
    index = faiss.IndexHNSWFlat(
        vectors.shape[1], M, faiss.METRIC_INNER_PRODUCT
    )
    index.hnsw.efConstruction = ef_construction
    index.add(vectors)
    index.hnsw.efSearch = ef_search
    del vectors
    return index, model.wv.index_to_key


def item2vec_top_k(session_aids, session_types, session_ts, model, index, idx_to_aid,
                    top_k=TOP_K, last_n=LAST_N):
    if not session_aids:
        return []

    scores = defaultdict(float)
    min_ts = min(session_ts)
    max_ts = max(session_ts)
    time_span = max(max_ts - min_ts, 1.0)

    for aid, event_type, ts in zip(session_aids, session_types, session_ts):
        type_weight = ACTION_WEIGHTS.get(event_type, 1.0)
        time_weight = (ts - min_ts) / time_span if time_span > 0 else 1.0
        scores[aid] += type_weight * time_weight

    recent_aids = session_aids[-last_n:]
    recent_types = session_types[-last_n:]
    recent_ts = session_ts[-last_n:]
    vectors = []
    weights = []
    for aid, event_type, ts in zip(recent_aids, recent_types, recent_ts):
        if aid in model.wv.key_to_index:
            type_weight = ACTION_WEIGHTS.get(event_type, 1.0)
            time_weight = (ts - min_ts) / time_span if time_span > 0 else 1.0
            vectors.append(model.wv[aid])
            weights.append(type_weight * time_weight)

    if vectors:
        weights = np.asarray(weights, dtype=np.float32)
        weight_sum = weights.sum()
        if weight_sum > 0:
            weights /= weight_sum
        else:
            weights[:] = 1.0 / len(weights)
        session_vector = np.average(vectors, axis=0, weights=weights)
        session_vector = np.asarray(session_vector, dtype=np.float32).reshape(1, -1)
        faiss.normalize_L2(session_vector)
        distances, indices = index.search(session_vector, 2 * top_k)

        for item_index, similarity in zip(indices[0], distances[0]):
            if item_index == -1:
                continue
            aid = idx_to_aid[item_index]
            neighbor_weight = max(float(similarity), 0.0)
            scores[aid] += neighbor_weight

    return [aid for aid, _ in sorted(scores.items(), key=lambda pair: pair[1], reverse=True)[:top_k]]


metric_counts = {
    split: {
        event_type: {'hits': 0, 'denominator': 0}
        for event_type in EVENT_TYPES
    }
    for split in ('train', 'validation')
}
evaluated_sessions = {'train': 0, 'validation': 0}
train_stride = max(1, math.ceil(train_sessions_count / MAX_TRAIN_EVAL_SESSIONS))
print(
    f'Оценка train: до {MAX_TRAIN_EVAL_SESSIONS:,} сессий '
    f'(каждая {train_stride}-я train-сессия). '
    f'Оценка validation: все {validation_sessions_count:,} сессий.'
)


def make_targets(future_events):
    targets = {event_type: set() for event_type in EVENT_TYPES}
    for event in future_events:
        event_type = event['type']
        aid = str(event['aid'])
        if event_type == 'clicks':
            if not targets['clicks']:
                targets['clicks'].add(aid)
        elif event_type in ('carts', 'orders'):
            targets[event_type].add(aid)
    return targets


def update_split_metrics(split_name, history, future, model_event_type,
                         model, index, idx_to_aid, processed_by_split):
    if not future:
        return

    processed_by_split[split_name] += 1
    if model_event_type == EVENT_TYPES[0]:
        evaluated_sessions[split_name] += 1

    target_set = make_targets(future)[model_event_type]
    denominator = min(TOP_K, len(target_set))
    if denominator:
        session_aids = [str(event['aid']) for event in history]
        session_types = [event['type'] for event in history]
        session_ts = [float(event['ts']) for event in history]
        predictions = item2vec_top_k(
            session_aids, session_types, session_ts,
            model, index, idx_to_aid,
        )
        metric_counts[split_name][model_event_type]['hits'] += len(
            set(predictions) & target_set
        )
        metric_counts[split_name][model_event_type]['denominator'] += denominator

    if (processed_by_split[split_name] and
            processed_by_split[split_name] % EVAL_PROGRESS_EVERY == 0):
        print(
            f'  {split_name}: обработано '
            f'{processed_by_split[split_name]:,} сессий'
        )


for model_event_type in EVENT_TYPES:
    model_path = OUT_DIR / f'item2vec_{model_event_type}.model'
    model = Word2Vec.load(str(model_path))
    index, idx_to_aid = build_item2vec_index(model)
    print(f'Оценка Item2Vec для группы {model_event_type}')

    processed_by_split = {'train': 0, 'validation': 0}
    train_session_offset = 0

    with open(TRAIN_PATH, 'r') as train_file:
        for session_index, line in enumerate(train_file):
            record = json.loads(line)
            events = record.get('events', [])
            train_events = [
                event for event in events
                if int(event['ts']) < VALIDATION_START_TS
            ]
            validation_events = [
                event for event in events
                if int(event['ts']) >= VALIDATION_START_TS
            ]

            if train_events:
                selected_for_train = train_session_offset % train_stride == 0
                if selected_for_train and len(train_events) >= 2:
                    split_idx = max(1, int(len(train_events) * 0.8))
                    update_split_metrics(
                        'train', train_events[:split_idx], train_events[split_idx:],
                        model_event_type, model, index, idx_to_aid, processed_by_split,
                    )
                train_session_offset += 1

            if validation_events:
                update_split_metrics(
                    'validation', train_events, validation_events,
                    model_event_type, model, index, idx_to_aid, processed_by_split,
                )

            if session_index and session_index % 1_000_000 == 0:
                print(f'  Прочитано записей-сессий: {session_index + 1:,}')

    del index, idx_to_aid, model
    gc.collect()

if evaluated_sessions['validation'] != validation_sessions_count:
    raise RuntimeError(
        'Оценены не все validation-сессии: '
        f'{evaluated_sessions["validation"]:,} из {validation_sessions_count:,}.'
    )

score_rows = []
for split_name in ('train', 'validation'):
    row = {'split': split_name, 'sessions_evaluated': evaluated_sessions[split_name]}
    recalls = {}
    for event_type in EVENT_TYPES:
        counts = metric_counts[split_name][event_type]
        recall = counts['hits'] / counts['denominator'] if counts['denominator'] else 0.0
        recalls[event_type] = recall
        row[f'{event_type}_hits'] = counts['hits']
        row[f'{event_type}_denominator'] = counts['denominator']
        row[f'{event_type}_recall_at_20'] = recall
    row['weighted_recall_at_20'] = sum(
        EVENT_WEIGHTS[event_type] * recalls[event_type]
        for event_type in EVENT_TYPES
    )
    score_rows.append(row)

output_path = OUT_DIR / 'item2vec_recall_at_20.csv'
fieldnames = list(score_rows[0])
with open(output_path, 'w', newline='') as output_file:
    writer = csv.DictWriter(output_file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(score_rows)

print('Recall@20 по группам и взвешенный итог:')
for row in score_rows:
    print(
        f"{row['split']}: "
        f"clicks={row['clicks_recall_at_20']:.5f} "
        f"({row['clicks_hits']:,}/{row['clicks_denominator']:,}), "
        f"carts={row['carts_recall_at_20']:.5f} "
        f"({row['carts_hits']:,}/{row['carts_denominator']:,}), "
        f"orders={row['orders_recall_at_20']:.5f} "
        f"({row['orders_hits']:,}/{row['orders_denominator']:,}), "
        f"weighted={row['weighted_recall_at_20']:.5f}; "
        f"sessions={row['sessions_evaluated']:,}"
    )
print(f'Файл со score: {output_path}')


Оценка train: до 100,000 сессий (каждая 111-я train-сессия). Оценка validation: все 5,323,084 сессий.
Оценка Item2Vec для группы clicks
  validation: обработано 25,000 сессий
  validation: обработано 50,000 сессий
  validation: обработано 75,000 сессий
  validation: обработано 100,000 сессий
  validation: обработано 125,000 сессий
  validation: обработано 150,000 сессий
  validation: обработано 175,000 сессий
  validation: обработано 200,000 сессий
  validation: обработано 225,000 сессий
  validation: обработано 250,000 сессий
  validation: обработано 275,000 сессий
  validation: обработано 300,000 сессий
  validation: обработано 325,000 сессий
  validation: обработано 350,000 сессий
  validation: обработано 375,000 сессий
  validation: обработано 400,000 сессий
  validation: обработано 425,000 сессий
  Прочитано записей-сессий: 1,000,001
  validation: обработано 450,000 сессий
  validation: обработано 475,000 сессий
  validation: обработано 500,000 сессий
  validation: обработано 525,